### Setup
Autoreload and imports

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# Imports
import sys, os
from pathlib import Path
from art.attacks.evasion import FastGradientMethod, ProjectedGradientDescent, SaliencyMapMethod, CarliniL2Method
from art.defences.trainer import AdversarialTrainer

sys.path.append(str(Path.cwd().parents[2]))

from utils.functions import get_windowed_data
from utils.notebook import get_model_classifier, clean_data_test, adv_test, FilenameLoader, get_filename_from_path

/opt/anaconda3/envs/reu/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Adversarial training

In [3]:
## Load checkpoint and data file paths
checkpoint_name, data_name, _ = FilenameLoader.rand_pos()

checkpoint_file= f"../../../saved_models/{checkpoint_name}"
data_file = f"../../../data/{data_name}"

In [4]:
## Load data
(x_train, y_train), (x_test, y_test), fed_dataset, scaler = get_windowed_data(data_file, 
                                                                      normalize=True, 
                                                                      train_perc=80)

In [ ]:
## From JSMA paper - iteratively trying to find optimal FGSM
# Adversarial training: fine-tune each vehicle's model on clean + FGSM examples, then re-evaluate
from art.defences.trainer import AdversarialTrainer
import torch 
import numpy as np 

# Training vars
adv_train_eps = 0.01
adv_train_epochs = 5
adv_train_ratio = 0.5 

x_train_np = x_train.numpy()
y_train_np = y_train.numpy()

# Init vars
name = get_filename_from_path(checkpoint_file)

done_eps = []
index = 10

# Condition 1: Fool model
# Condition 2: doesn't decrease original model performance that much
while not adv_train_eps in done_eps and len(done_eps) < 50:
    model, classifier = get_model_classifier(checkpoint_file)

    index +=1
    print(f"=== Adversarial training: {name} | eps:  {adv_train_eps} ===")
    print(f"done eps: {done_eps}")

    done_eps.append(adv_train_eps)
    
    adv_save_dir = f"../fed/data-test/iterative/iter-{index}"
    os.makedirs(adv_save_dir, exist_ok=True)

    # Get clean baseline 
    print("> Before Clean Test")
    before_clean_out = clean_data_test(
        model, classifier, x_test, y_test, 
        checkpoint_file=checkpoint_file, data_file=data_file,
        save_path=adv_save_dir,
        filename=f"before_clean_advtrained_{name}.json",
        save_results=True
    )

    # Check condition 1
    print ("> Before Adv Test")
    before_adv_out = adv_test(
        classifier, x_test, y_test, 
        checkpoint_file=checkpoint_file, data_file=data_file,
        end_index=len(y_test.numpy()),
        path=adv_save_dir,
        filename=f"before_adv_eps_{adv_train_eps}_advtrained_{name}.json",
        Attack=FastGradientMethod,
        eps=adv_train_eps,
    )

    print("> Check Condition 1")
    if before_adv_out['metrics']['f1'] > 0.5:
        print("[!!!] Condition 1: Fool model FAILED (adv f1 > 0.5)")
        adv_train_eps = adv_train_eps + 0.01
        print("New eps:", adv_train_eps)

        continue
    print("[O] Condition 1 Good")
    
    # Run adv training
    print("> Running Adv Training")
    attack = FastGradientMethod(classifier, eps=adv_train_eps)
    trainer = AdversarialTrainer(classifier, attacks=attack, ratio=adv_train_ratio)
    trainer.fit(x_train_np, y_train_np, nb_epochs=adv_train_epochs)

    # Re-evaluate after adversarial training
    print("> After Clean Test")
    after_clean_out = clean_data_test(
        model, classifier, x_test, y_test, 
        checkpoint_file=checkpoint_file, data_file=data_file,
        save_path=adv_save_dir,
        filename=f"after_clean_advtrained_{name}.json",
        save_results=True
    )

    print("> After Adv Test")
    # do a +- 0.2
    lo = max(0.01, adv_train_eps - 0.02)
    hi = min(1.0, adv_train_eps + 0.02)
    lo_pct = int(round(lo * 100))
    hi_pct = int(round(hi * 100))

    after_adv_f1 = []
    for i in range(lo_pct, hi_pct):
        eps = float(i/100)
        after_adv_out = adv_test(
            classifier, x_test, y_test, 
            checkpoint_file=checkpoint_file, data_file=data_file,
            end_index=len(y_test.numpy()),
            path=adv_save_dir,
            filename=f"after_adv_eps_{eps}_advtrained_{name}.json",
            Attack=FastGradientMethod,
            eps=eps,
        )
        after_adv_f1.append(after_adv_out["metrics"]["f1"])
    

    # Save the adversarially-trained weights (loadable via utils.functions.load_model_checkpoint)
    ckpt_out = f"{adv_save_dir}/advtrained_{name}.ckpt"
    torch.save(model.learner.state_dict(), ckpt_out)
    print(f"Saved adversarially-trained checkpoint to {ckpt_out}")

    print("> Check Condition 2")
    # Check Condition 2: if the 
    if before_clean_out["wrapper"]["f1"] - after_clean_out["wrapper"]["f1"] > 0.1:
        print("[!!!] Condition 2: Maintain F1 FAILED (differense is greater than 0.1)")
        adv_train_eps = adv_train_eps - 0.01
        print("New eps:", adv_train_eps)
    else:
        print("[O] Condition 2 Good")
    
    print("> Check Condition 3: adv after is good")


    if all(f1 >= 0.8 for f1 in after_adv_f1):
        print("[O] Condition 3 Good")
    else:
        print("[!!!] Condition 3: Improve Adv FAILED (clean f1 < 0.8)")
        print("Not changing eps, but be aware of this")

    

GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Checkpoint path exists!
=== Adversarial training: RandomPos-final | eps:  0.01 ===
done eps: []
> Before Clean Test
Saved to ../fed/data-test/iterative/iter-11/before_clean_advtrained_RandomPos-final.json
> Before Adv Test
=== Attack: FastGradientMethod, kwargs: {'eps': 0.01} ===


GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Accuracy:           0.9822
Precision:          0.9984
Recall:             0.9416
F1:                 0.9692
ASR (FNR):          0.0584
False Positive Rate:0.0006
TP=349045, TN=876489, FP=551, FN=21655
Time elapsed:       15.15s
Saved metrics to ../fed/data-test/iterative/iter-11/before_adv_eps_0.01_advtrained_RandomPos-final.json
> Check Condition 1
[!!!] Condition 1: Fool model FAILED (adv f1 > 0.5)
New eps: 0.02
Checkpoint path exists!
=== Adversarial training: RandomPos-final | eps:  0.02 ===
done eps: [0.01]
> Before Clean Test
Saved to ../fed/data-test/iterative/iter-12/before_clean_advtrained_RandomPos-final.json
> Before Adv Test
=== Attack: FastGradientMethod, kwargs: {'eps': 0.02} ===


GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


...

Accuracy:           0.6869
Precision:          0.4841
Recall:             0.8221
F1:                 0.6094
ASR (FNR):          0.1779
False Positive Rate:0.3702
TP=304739, TN=552328, FP=324712, FN=65961
Time elapsed:       16.45s
Saved metrics to ../fed/data-test/iterative/iter-18/before_adv_eps_0.08_advtrained_RandomPos-final.json
> Check Condition 1
[!!!] Condition 1: Fool model FAILED (adv f1 > 0.5)
New eps: 0.09
Checkpoint path exists!
=== Adversarial training: RandomPos-final | eps:  0.09 ===
done eps: [0.01, 0.02, 0.03, 0.04, 0.05, 0.060000000000000005, 0.07, 0.08]
> Before Clean Test
Saved to ../fed/data-test/iterative/iter-19/before_clean_advtrained_RandomPos-final.json
> Before Adv Test
=== Attack: FastGradientMethod, kwargs: {'eps': 0.09} ===


GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Accuracy:           0.5390
Precision:          0.3721
Recall:             0.8025
F1:                 0.5084
ASR (FNR):          0.1975
False Positive Rate:0.5723
TP=297476, TN=375087, FP=501953, FN=73224
Time elapsed:       15.30s
Saved metrics to ../fed/data-test/iterative/iter-19/before_adv_eps_0.09_advtrained_RandomPos-final.json
> Check Condition 1
[!!!] Condition 1: Fool model FAILED (adv f1 > 0.5)
New eps: 0.09999999999999999
Checkpoint path exists!
=== Adversarial training: RandomPos-final | eps:  0.09999999999999999 ===
done eps: [0.01, 0.02, 0.03, 0.04, 0.05, 0.060000000000000005, 0.07, 0.08, 0.09]
> Before Clean Test
Saved to ../fed/data-test/iterative/iter-20/before_clean_advtrained_RandomPos-final.json
> Before Adv Test
=== Attack: FastGradientMethod, kwargs: {'eps': 0.09999999999999999} ===
Accuracy:           0.3582
Precision:          0.2870
Recall:             0.7814
F1:                 0.4198
ASR (FNR):          0.2186
False Positive Rate:0.8207
TP=289683, TN=157230, F

Adversarial training epochs: 100%|██████████| 5/5 [04:11<00:00, 50.29s/it]


> After Clean Test
Saved to ../fed/data-test/iterative/iter-20/after_clean_advtrained_RandomPos-final.json
> After Adv Test
=== Attack: FastGradientMethod, kwargs: {'eps': 0.08} ===
Accuracy:           0.3464
Precision:          0.3103
Recall:             0.9816
F1:                 0.4716
ASR (FNR):          0.0184
False Positive Rate:0.9221
TP=363880, TN=68333, FP=808707, FN=6820
Time elapsed:       15.27s
Saved metrics to ../fed/data-test/iterative/iter-20/after_adv_eps_0.08_advtrained_RandomPos-final.json
=== Attack: FastGradientMethod, kwargs: {'eps': 0.09} ===
Accuracy:           0.8062
Precision:          0.6077
Recall:             0.9810
F1:                 0.7505
ASR (FNR):          0.0190
False Positive Rate:0.2676
TP=363662, TN=642320, FP=234720, FN=7038
Time elapsed:       15.18s
Saved metrics to ../fed/data-test/iterative/iter-20/after_adv_eps_0.09_advtrained_RandomPos-final.json
=== Attack: FastGradientMethod, kwargs: {'eps': 0.1} ===
Accuracy:           0.9926
Precision: 

In [ ]:
## Find lowerbound threshold - lowest eps that still
lo_eps = 0.07 # fgsm, from the previously done eps sweep, f1 = 0.7079755805153897 < 0.77


In [ ]:
## Find upperbound threshold - highest eps that starts making benign wrong

import json, time, torch
# When f1 is at least 20 pts down from og (so 0.77)

# Testing if greater epochs, with half/half split of ratio will work better
# make eps really high to see if there even is an upper bound

# Defined
clean_f1 = 0.976470669379591
threshold_diff = 0.2
working_eps = 0.5

# Training Hyperparams
adv_train_epochs = 30
adv_train_ratio = 0.5

# Var definitions
x_train_np = x_train.numpy()
y_train_np = y_train.numpy()
name = get_filename_from_path(checkpoint_file)
index = 1

while index < 100 and working_eps < 1:
    start = time.time()
    print(f"\n\n>>> New eps: {working_eps}")
    # Reload classifier
    index = index + 1
    model, classifier = get_model_classifier(checkpoint_file)
    adv_save_dir = f"../fed/data-test/high-eps-iterative-ratio-0.5-epochs-30/iter-{index}"
    os.makedirs(adv_save_dir, exist_ok=True)
    print(f"Saving to: {adv_save_dir}")

    ## Adv train model
    print("> Running Adv Training")
    attack = FastGradientMethod(classifier, eps=working_eps)
    trainer = AdversarialTrainer(classifier, attacks=attack, ratio=adv_train_ratio)
    trainer.fit(x_train_np, y_train_np, nb_epochs=adv_train_epochs)

    # Save model
    ckpt_out = f"{adv_save_dir}/advtrained_{name}.ckpt"
    torch.save(model.learner.state_dict(), ckpt_out)
    print(f"Saved adversarially-trained checkpoint to {ckpt_out}")

    ## Test model
    after_clean_out = clean_data_test(
        model, classifier, x_test, y_test, 
        checkpoint_file=checkpoint_file, data_file=data_file,
        save_path=adv_save_dir,
        filename=f"after_clean_advtrained_{name}.json",
        save_results=True
    )

    # Save config
    config = {
        "eps": working_eps,
        "adv_train_epochs": adv_train_epochs,
        "adv_train_ratio": adv_train_ratio,
        "clean_f1_baseline": clean_f1,
        "threshold_diff": threshold_diff,
        "index": index,
        "checkpoint_file": checkpoint_file,
        "name": name,
        "timeElapsedSec": (time.time() - start)
    }

    with open(f"{adv_save_dir}/config.json", "w") as f:
        json.dump(config, f, indent=4)

    # Check if f1 is below
    adv_f1 = after_clean_out["wrapper"]["f1"]
    if (adv_f1 < clean_f1 - threshold_diff): 
        print(f"BREAKING WITH EPS {working_eps}")
        break
    else:
        print(f"f1 = {adv_f1} >  {clean_f1 - threshold_diff}")
        working_eps = round(working_eps + 0.01, 2)
        print(f"Increasing eps to {working_eps}")

print("goodbye world")


    
    

"""
- Start at ~ 0.2 eps
- Adv train model on the # of eps (save the model)
- Run clean test (and SAVE it)
- Check if it's below the threshold

* Can try just doing 10 points.. later (or just see from data)
"""

GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
/opt/anaconda3/envs/reu/lib/python3.11/site-packages/pytorch_lightning/trainer/setup.py:175: GPU available but not used. You can set it by doing `Trainer(accelerator='gpu')`.
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.




>>> New eps: 0.2
Checkpoint path exists!
Saving to: ../fed/data-test/high-eps-iterative-ratio-0.5-epochs-30/iter-0
> Running Adv Training


Adversarial training epochs: 100%|██████████| 30/30 [31:12<00:00, 62.43s/it]


Saved adversarially-trained checkpoint to ../fed/data-test/high-eps-iterative-ratio-0.5-epochs-30/iter-0/advtrained_RandomPos-final.ckpt


GPU available: True (mps), used: False
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.


Saved to ../fed/data-test/high-eps-iterative-ratio-0.5-epochs-30/iter-0/after_clean_advtrained_RandomPos-final.json
f1 = 0.9948869526616808 >  0.7764706693795911
Increasing eps to 0.21


>>> New eps: 0.21
Checkpoint path exists!
Saving to: ../fed/data-test/high-eps-iterative-ratio-0.5-epochs-30/iter-1
> Running Adv Training


Adversarial training epochs:  50%|█████     | 15/30 [16:43<16:43, 66.88s/it]


KeyboardInterrupt: 

In [16]:
min(3, 1)

1